# 🧠 TechMentor — Dual-Model Technical Tutor

**Week 1 Exercise — Community Contribution**

A creative take on the technical Q&A tutor that goes beyond the basics:

- 💬 **Interactive input** — ask any technical question at runtime
- 🎓 **Explanation depth selector** — Beginner / Intermediate / Expert
- ⚡ **GPT-4o-mini** answers with **live streaming**
- 🦙 **Llama 3.2** (local via Ollama) answers with **streaming**
- 📊 **Side-by-side comparison** — both models, same question
- 🏆 **AI Verdict** — GPT judges which answer was stronger for your chosen depth

In [ ]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, update_display, HTML
import ollama

In [ ]:
# constants

MODEL_GPT   = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

DEPTH_PROMPTS = {
    "beginner":     "Explain simply, avoid jargon, use analogies and relatable examples.",
    "intermediate": "Give a thorough explanation with code examples and best practices.",
    "expert":       "Give an in-depth technical explanation covering edge cases, internals, and performance implications.",
}

In [ ]:
# set up environment

load_dotenv()
client = OpenAI()

In [ ]:
# Interactive question + depth selector

print("🧠 TechMentor — Ask anything about Python, LLMs, data science, or software engineering\n")

question = input("Your question: ").strip()
if not question:
    question = 'Please explain what this code does and why:\nyield from {book.get("author") for book in books if book.get("author")}'
    print("(No input — using default question)")

depth = input("\nExplanation depth [beginner / intermediate / expert] (default: intermediate): ").strip().lower()
if depth not in DEPTH_PROMPTS:
    depth = "intermediate"

print(f"\n✅ Depth set to: {depth.capitalize()}")

In [ ]:
# Build prompts

system_prompt = (
    "You are TechMentor — a sharp, friendly technical tutor specialising in Python, "
    "software engineering, data science, and LLMs. "
    f"{DEPTH_PROMPTS[depth]} "
    "Use clear markdown formatting with sections, bullet points, and code blocks."
)

user_prompt = f"Please explain the following in detail:\n\n{question}"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": user_prompt},
]

In [ ]:
# GPT-4o-mini — streamed answer

display(HTML("""
<div style="background:#0d1117;color:#58a6ff;padding:10px 16px;border-radius:6px;
            font-family:monospace;font-size:14px;margin-bottom:6px;border-left:4px solid #58a6ff">
  ⚡ <b>GPT-4o-mini</b> &nbsp;<span style="color:#8b949e">(OpenAI · streaming)</span>
</div>
"""))

stream = client.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)

gpt_response = ""
gpt_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    gpt_response += chunk.choices[0].delta.content or ""
    update_display(Markdown(gpt_response), display_id=gpt_handle.display_id)

In [ ]:
# Llama 3.2 — local Ollama with streaming

display(HTML("""
<div style="background:#161b22;color:#3fb950;padding:10px 16px;border-radius:6px;
            font-family:monospace;font-size:14px;margin-bottom:6px;border-left:4px solid #3fb950">
  🦙 <b>Llama 3.2</b> &nbsp;<span style="color:#8b949e">(Ollama · local · streaming)</span>
</div>
"""))

llama_response = ""

try:
    llama_handle = display(Markdown(""), display_id=True)
    for chunk in ollama.chat(model=MODEL_LLAMA, messages=messages, stream=True):
        llama_response += chunk["message"]["content"]
        update_display(Markdown(llama_response), display_id=llama_handle.display_id)
except Exception as e:
    display(Markdown(f"⚠️ Ollama unavailable: `{e}`  \nEnsure Ollama is running and `{MODEL_LLAMA}` is pulled (`ollama pull llama3.2`)."))

In [ ]:
# 🏆 AI Verdict — GPT judges both answers

display(HTML("""
<div style="background:#21262d;color:#f0883e;padding:10px 16px;border-radius:6px;
            font-family:monospace;font-size:14px;margin-bottom:6px;border-left:4px solid #f0883e">
  🏆 <b>AI Verdict</b> &nbsp;<span style="color:#8b949e">(GPT-4o-mini compares both answers)</span>
</div>
"""))

if llama_response:
    verdict_messages = [
        {"role": "system", "content": "You are a concise, fair technical judge. Be direct and specific."},
        {"role": "user",   "content": (
            f"Question asked: {question}\n"
            f"Target audience: {depth}\n\n"
            f"--- Answer A (GPT-4o-mini) ---\n{gpt_response}\n\n"
            f"--- Answer B (Llama 3.2) ---\n{llama_response}\n\n"
            "Give a 3-bullet verdict: one strength of A, one strength of B, "
            "and which answer better suits the target audience and why."
        )},
    ]
    verdict_stream = client.chat.completions.create(model=MODEL_GPT, messages=verdict_messages, stream=True)
    verdict = ""
    verdict_handle = display(Markdown(""), display_id=True)
    for chunk in verdict_stream:
        verdict += chunk.choices[0].delta.content or ""
        update_display(Markdown(verdict), display_id=verdict_handle.display_id)
else:
    display(Markdown("*Verdict skipped — Llama 3.2 was unavailable.*"))